In [21]:
import json
import pandas as pd
from pathlib import Path
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

In [22]:
ROOT_PATH = "/home/stefan/ioai-prep/kits/contest/magic"
TEST_DIR = f"{ROOT_PATH}/test"
SUBMISSION_PATH = f"{ROOT_PATH}/submission.csv"
MODEL_NAME = "openai/clip-vit-large-patch14"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data

In [23]:
def load_test_data(test_dir: str):
    test_records = []
    for meta_path in Path(test_dir).glob("*/metadata.json"):
        with open(meta_path) as f:
            d = json.load(f)
        
        word_choices = list(dict.fromkeys(d["word_choices"]))
        
        rec = {
            "id": meta_path.parent.name,
            "image_path": str(meta_path.parent / "image.png"),
            "word_choices": word_choices,
        }
        test_records.append(rec)
    
    return test_records

In [24]:
def predict_top5_ensemble(
    record: dict, models: list, processors: list, device: torch.device
):
    image = Image.open(record["image_path"]).convert("RGB")
    words = record["word_choices"]

    logits_ensemble = None
    for model, processor in zip(models, processors):
        inputs = processor(
            text=words,
            images=image,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits_per_image[0].cpu().numpy()

        logits_ensemble = (
            logits if logits_ensemble is None else logits_ensemble + logits
        )

    logits_ensemble /= len(models)
    top5_indices = logits_ensemble.argsort()[::-1][:5]
    return [words[i] for i in top5_indices]

In [25]:
def analyze_training_data(train_dir: str):
    word_frequency = {}
    for meta_path in Path(train_dir).glob("*/metadata.json"):
        with open(meta_path) as f:
            d = json.load(f)
        for word in d["correct_words"]:
            word_frequency[word] = word_frequency.get(word, 0) + 1
    return word_frequency

In [26]:
def predict_top5(
    record: dict, models: list, processors: list, device: torch.device, word_freq: dict
):
    image = Image.open(record["image_path"]).convert("RGB")
    words = record["word_choices"]

    logits_ensemble = None
    for model, processor in zip(models, processors):
        inputs = processor(
            text=words,
            images=image,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits_per_image[0].cpu().numpy()

        logits_ensemble = (
            logits if logits_ensemble is None else logits_ensemble + logits
        )

    logits_ensemble /= len(models)

    # Apply frequency boost
    for i, word in enumerate(words):
        if word_freq.get(word, 0) > 1:
            logits_ensemble[i] *= 1.2

    top5_indices = logits_ensemble.argsort()[::-1][:5]
    return [words[i] for i in top5_indices]

In [27]:
print("Loading models...")
model_names = ["openai/clip-vit-large-patch14", "openai/clip-vit-base-patch32"]
models = [CLIPModel.from_pretrained(name).to(DEVICE) for name in model_names]
processors = [CLIPProcessor.from_pretrained(name) for name in model_names]

print("Analyzing training data...")
word_freq = analyze_training_data(f"{ROOT_PATH}/train")

test_records = load_test_data(TEST_DIR)
print(f"Found {len(test_records)} test samples")

print("Generating predictions...")
results = []
for idx, rec in enumerate(tqdm(test_records), 1):
    top5_words = predict_top5(rec, models, processors, DEVICE, word_freq)
    results.append(
        {
            "ID": rec["id"],
            "word1": top5_words[0],
            "word2": top5_words[1],
            "word3": top5_words[2],
            "word4": top5_words[3],
            "word5": top5_words[4],
        }
    )

Loading models...
Analyzing training data...
Found 100 test samples
Generating predictions...


100%|██████████| 100/100 [00:08<00:00, 12.38it/s]


In [28]:
submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")

Submission saved to: /home/stefan/ioai-prep/kits/contest/magic/submission.csv


In [29]:
submission_df.head()

,ID,word1,word2,word3,word4,word5
0,00078,hippo,tomato,shapeshifter,ritualcircle,hammer
1,00022,projector,desk,machine,spellbook,conjuration
2,00050,watercolor,aloe vera,onion,cauliflower,lemon
3,00009,lynx,tablet,cat,book,peas
4,00013,iris,scissors,projector,wreath,vase
